In [1]:
import subprocess
subprocess.run([
    "pip", "install",
    "qdrant-client",
    "sentence-transformers",
    "transformers",
    "torch",
    "Pillow",
    "tqdm",
    "--quiet"
])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 8.1 MB/s eta 0:00:00


CompletedProcess(args=['pip', 'install', 'qdrant-client', 'sentence-transformers', 'transformers', 'torch', 'Pillow', 'tqdm', '--quiet'], returncode=0)

In [2]:
import os
import json
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
 
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    OptimizersConfigDiff, HnswConfigDiff
)
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
 
# ── Config ─────────────────────────────────────────────────
INPUT_DIR       = Path("/kaggle/input/datasets/vudathab/source-data/embedding_source_data")
METADATA_FILE   = INPUT_DIR / "metadata.jsonl"
IMAGES_DIR      = INPUT_DIR / "images"
QDRANT_PATH     = Path("/kaggle/working/qdrant_storage")
 
COLLECTION_NAME = "arxiv_rag"
BATCH_SIZE      = 64
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
 
print(f"Device        : {DEVICE}")
print(f"Metadata      : {METADATA_FILE.exists()}")
print(f"Images dir    : {IMAGES_DIR.exists()}")
print(f"Images count  : {len(list(IMAGES_DIR.iterdir()))}")
 
# count record types
text_count = figure_count = 0
with open(METADATA_FILE) as f:
    for line in f:
        rec = json.loads(line)
        if rec["record_type"] == "text":
            text_count += 1
        else:
            figure_count += 1
 
print(f"\nText chunks   : {text_count:,}")
print(f"Figures       : {figure_count:,}")
print(f"Total         : {text_count + figure_count:,}")


Device        : cuda
Metadata      : True
Images dir    : True
Images count  : 226

Text chunks   : 998
Figures       : 226
Total         : 1,224


In [3]:
print("\n[+] Loading CLIP...")
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("    CLIP ready — 512-dim")
 
print("\n[+] Loading BGE...")
bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=DEVICE)
print("    BGE ready  — 768-dim")


[+] Loading CLIP...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

    BGE ready  — 768-dim


In [4]:
def embed_images(image_paths: list) -> np.ndarray:
    """
    Embed images with CLIP. Returns (N, 512) float32 array.
 
    Fix 1: Resize to 224x224 before loading into memory
           CLIP always processes at 224x224 anyway — resizing
           early reduces RAM usage for high-res figures
 
    Fix 2: Handle newer transformers returning output object
           instead of plain tensor from get_image_features()
    """
    images = []
    for path in image_paths:
        try:
            img = Image.open(path).convert("RGB")
            img = img.resize((224, 224), Image.LANCZOS)  # FIX 1
            images.append(img)
        except Exception:
            images.append(Image.new("RGB", (224, 224), color=0))
 
    with torch.no_grad():
        inputs = clip_processor(
            images=images,
            return_tensors="pt",
            padding=True
        ).to(DEVICE)
 
        features = clip_model.get_image_features(**inputs)
 
        # FIX 2: extract tensor if output is wrapped object
        if not isinstance(features, torch.Tensor):
            features = features.pooler_output
 
        # L2 normalize for cosine similarity
        features = features / features.norm(dim=-1, keepdim=True)
 
    return features.cpu().numpy().astype(np.float32)
 
 
def embed_texts(texts: list) -> np.ndarray:
    """
    Embed texts with BGE. Returns (N, 768) float32 array.
    BGE prefix improves passage retrieval quality.
    We embed context_text (header + chunk) for full provenance.
    """
    prefixed = [
        f"Represent this sentence for searching relevant passages: {t}"
        for t in texts
    ]
    return bge_model.encode(
        prefixed,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    ).astype(np.float32)

In [5]:
QDRANT_PATH.mkdir(parents=True, exist_ok=True)
client = QdrantClient(path=str(QDRANT_PATH))
 
existing = [c.name for c in client.get_collections().collections]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print(f"[+] Deleted existing '{COLLECTION_NAME}'")
 
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "image_vector": VectorParams(size=512, distance=Distance.COSINE),
        "text_vector" : VectorParams(size=768, distance=Distance.COSINE),
    },
    hnsw_config=HnswConfigDiff(m=16, ef_construct=200),
    optimizers_config=OptimizersConfigDiff(indexing_threshold=5000)
)
print(f"[+] Collection '{COLLECTION_NAME}' created")
print(f"    image_vector : 512-dim COSINE")
print(f"    text_vector  : 768-dim COSINE")

[+] Collection 'arxiv_rag' created
    image_vector : 512-dim COSINE
    text_vector  : 768-dim COSINE


In [6]:
print("\n[+] Loading records...")
records = []
with open(METADATA_FILE) as f:
    for line in f:
        records.append(json.loads(line.strip()))
 
print(f"    Loaded: {len(records):,}")
 
# FIX: Windows backslash paths on Linux
# Path("data\\extracted\\images\\file.png").name on Linux
# returns "data\\extracted\\images\\file.png" (whole thing)
# because Linux doesn't treat \ as separator
# Solution: replace \ with / first, then split
for r in records:
    if r.get("image_path"):
        raw      = r["image_path"]
        filename = raw.replace("\\", "/").split("/")[-1]
        r["kaggle_image_path"] = str(IMAGES_DIR / filename)
    else:
        r["kaggle_image_path"] = None
 
# verify fix
print("\n    Path fix verification (first 3 figures):")
for r in [r for r in records if r["record_type"] == "figure"][:3]:
    p = r["kaggle_image_path"]
    print(f"      {Path(p).name} → exists: {Path(p).exists()}")


[+] Loading records...
    Loaded: 1,224

    Path fix verification (first 3 figures):
      1706.03762_502f570a0547.png → exists: True
      1706.03762_3dc7af7ae472.png → exists: True
      1706.03762_9437844a1dc0.png → exists: True


In [7]:
ZERO_IMAGE_VEC = np.zeros(512, dtype=np.float32)
failed         = 0
total_batches  = (len(records) + BATCH_SIZE - 1) // BATCH_SIZE
 
print(f"\n[+] Embedding {len(records):,} records...")
print(f"    Batches : {total_batches}")
print(f"    Device  : {DEVICE}\n")
 
for batch_idx in tqdm(range(total_batches), desc="  Embedding"):
    start = batch_idx * BATCH_SIZE
    end   = min(start + BATCH_SIZE, len(records))
    batch = records[start:end]
    n     = len(batch)
 
    text_idx   = [i for i, r in enumerate(batch) if r["record_type"] == "text"]
    figure_idx = [i for i, r in enumerate(batch) if r["record_type"] == "figure"]
 
    image_vecs = np.tile(ZERO_IMAGE_VEC, (n, 1))
    text_vecs  = np.zeros((n, 768), dtype=np.float32)
 
    if text_idx:
        txts = [batch[i]["context_text"] for i in text_idx]
        vecs = embed_texts(txts)
        for j, idx in enumerate(text_idx):
            text_vecs[idx] = vecs[j]
 
    if figure_idx:
        paths    = [batch[i]["kaggle_image_path"] for i in figure_idx]
        img_vecs = embed_images(paths)
        for j, idx in enumerate(figure_idx):
            image_vecs[idx] = img_vecs[j]
 
        caps     = [batch[i]["context_text"] for i in figure_idx]
        cap_vecs = embed_texts(caps)
        for j, idx in enumerate(figure_idx):
            text_vecs[idx] = cap_vecs[j]
 
    points = []
    for i, rec in enumerate(batch):
        points.append(PointStruct(
            id     = start + i,
            vector = {
                "image_vector": image_vecs[i].tolist(),
                "text_vector" : text_vecs[i].tolist(),
            },
            payload = {
                "record_id"  : rec["id"],
                "record_type": rec["record_type"],
                "arxiv_id"   : rec["arxiv_id"],
                "paper_title": rec["paper_title"],
                "category"   : rec["category"],
                "section"    : rec["section"],
                "text"       : rec["text"],
                "chunk_index": rec.get("chunk_index", 0),
                "chunk_total": rec.get("chunk_total", 1),
                "word_count" : rec.get("word_count", 0),
                "image_path" : rec.get("image_path"),
                "caption"    : rec.get("caption", ""),
                "figure_num" : rec.get("figure_num", ""),
                "page"       : rec.get("page", 0),
            }
        ))
 
    try:
        client.upsert(
            collection_name=COLLECTION_NAME,
            points=points,
            wait=True,
        )
    except Exception as e:
        tqdm.write(f"\n  Batch {batch_idx} failed: {e}")
        failed += n
 
print(f"\n    Indexed : {len(records) - failed:,}")
print(f"    Failed  : {failed}")


[+] Embedding 1,224 records...
    Batches : 20
    Device  : cuda



  Embedding: 100%|██████████| 20/20 [00:42<00:00,  2.14s/it]


    Indexed : 1,224
    Failed  : 0


In [8]:

import shutil

print("[+] Zipping Qdrant storage...")

# zip the entire qdrant_storage folder
zip_path = "/kaggle/working/qdrant_backup"
shutil.make_archive(
    zip_path,           # output path (without .zip)
    "zip",              # format
    "/kaggle/working",  # root directory
    "qdrant_storage"    # folder to zip
)

print(f"\n{'='*60}")
print(f"  DOWNLOAD NOW FROM KAGGLE OUTPUT TAB:")
print(f"  qdrant_backup.zip")
print(f"{'='*60}")

# verify zip size
import os
size_mb = os.path.getsize(zip_path + ".zip") / (1024*1024)
print(f"\n  Zip size: {size_mb:.1f} MB")

[+] Zipping Qdrant storage...

  DOWNLOAD NOW FROM KAGGLE OUTPUT TAB:
  qdrant_backup.zip

  Zip size: 5.4 MB


In [9]:
info = client.get_collection(COLLECTION_NAME)
print(f"\n[+] Collection status : {info.status}")
print(f"    Total points      : {info.points_count:,}")
 
# text search test
print(f"\n[+] Text search test:")
test_rec = next(r for r in records if r["record_type"] == "text")
test_vec = embed_texts([test_rec["context_text"]])[0]
 
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=test_vec.tolist(),
    using="text_vector",
    limit=3,
    with_payload=True,
).points
 
print(f"    Query: '{test_rec['paper_title']}' — {test_rec['section']}")
for i, r in enumerate(results):
    print(f"    [{i+1}] {r.score:.4f} | {r.payload['paper_title']} | {r.payload['section']}")
 
# image search test with deduplication
print(f"\n[+] Image search test (deduplicated):")
test_fig = next(
    (r for r in records
     if r["record_type"] == "figure"
     and r.get("kaggle_image_path")
     and Path(r["kaggle_image_path"]).exists()
     and not r.get("caption", "").startswith("Figure from")),
    None
)
 
if test_fig:
    img_vec = embed_images([test_fig["kaggle_image_path"]])[0]
    raw_results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=img_vec.tolist(),
        using="image_vector",
        limit=20,          # fetch more, then deduplicate
        with_payload=True,
    ).points
 
    # deduplicate by arxiv_id + figure_num + page
    seen = set()
    unique = []
    for r in raw_results:
        key = (
            r.payload["arxiv_id"],
            r.payload.get("figure_num", ""),
            r.payload.get("page", 0)
        )
        if key not in seen:
            seen.add(key)
            unique.append(r)
 
    print(f"    Query  : '{test_fig['paper_title']}'")
    print(f"    Caption: '{test_fig['caption'][:80]}'")
    print(f"\n    Top 5 unique results:")
    for i, r in enumerate(unique[:5]):
        print(f"    [{i+1}] {r.score:.4f} | {r.payload['paper_title']}")
        print(f"           {r.payload.get('caption','')[:70]}")
else:
    print("    No figure with real caption found")


[+] Collection status : green
    Total points      : 1,224

[+] Text search test:
    Query: 'Attention Is All You Need' — Abstract
    [1] 1.0000 | Attention Is All You Need | Abstract
    [2] 0.9203 | Attention Is All You Need | 7 Conclusion
    [3] 0.8958 | Attention Is All You Need | Attention Is All You Need / 1 Introduction

[+] Image search test (deduplicated):
    Query  : 'Attention Is All You Need'
    Caption: 'Figure 1: The Transformer - model architecture.'

    Top 5 unique results:
    [1] 1.0000 | Attention Is All You Need
           Figure 1: The Transformer - model architecture.
    [2] 0.8594 | Drag Your GAN: Interactive Point-based Manipulation on Generative Image
           Figure 3: Tasks groups included in MULTIS multimodal instruction tunin
    [3] 0.8503 | FlashAttention-2: Faster Attention with Better Parallelism
           Figure 1: Diagram of how FlashAttention forward pass is performed, whe
    [4] 0.8472 | Mamba: Linear-Time Sequence Modeling with Select